# SANA Sign — Standalone Optuna Hyperparameter Search

**Purpose:** Find the best hyperparameters for the Visual Encoder before the long multi-zip training run.

**This notebook is completely independent of the main training notebook.**

- Samples **10,000 clips** randomly from whichever zip is attached
- Runs **20 Optuna trials**, each training for 3 short epochs
- Estimated total time on T4 GPU: **3-5 hours**
- Outputs best CONFIG values to paste into Khizer's training notebook

### Setup
Attach two datasets in the Kaggle sidebar before running:
1. Your LINDAT zip file (the `content` blob)
2. Your `YT.translations.all.json` caption file

In [ ]:
# ══════════════════════════════════════════════════════════
# ONLY EDIT THIS CELL
# ══════════════════════════════════════════════════════════

CONFIG = {
    # ── Paths (update these to match your Kaggle dataset names) ──
    "ZIP_PATH":          "/kaggle/input/yt-asl-zip1/content",
    "TRANSLATIONS_PATH": "/kaggle/input/yt-asl-captions/YT.translations.all.json",

    # ── Architecture (DO NOT CHANGE — fixed by mT5-small) ──
    "MT5_MODEL_NAME":    "google/mt5-small",
    "D_MODEL":           512,     # Must match mT5-small hidden size — not tunable
    "NUM_HEADS":         8,       # Must divide D_MODEL evenly — not tunable
    "MAX_SEQ_LEN":       300,
    "NUM_KEYPOINTS":     104,
    "KEYPOINT_DIM":      2,
    "INPUT_DIM":         208,     # 104 * 2 — fixed by MediaPipe
    "MAX_TARGET_LENGTH": 128,
    "FP16":              True,
    "WEIGHT_DECAY":      0.01,
    "LORA_ENABLED":      False,

    # ── Optuna search settings ──
    "OPTUNA_SAMPLE_SIZE": 10000,  # clips to sample from the zip
    "OPTUNA_EPOCHS":      3,      # epochs per trial (signal without overfitting)
    "OPTUNA_N_TRIALS":    20,     # number of hyperparameter combinations to try
    "OPTUNA_SEED":        99,

    # ── Keypoint schema fallbacks (from Khizer's verified notebook) ──
    "LEFT_SHOULDER_IDX":  11,
    "RIGHT_SHOULDER_IDX": 12,
    "LEFT_HIP_IDX":       23,
    "RIGHT_HIP_IDX":      24,
    "SELECTED_FACE_INDICES": [
        0, 1, 13, 14, 17, 33, 37, 39, 40, 61, 63, 66, 70, 78, 84, 105, 181,
        263, 267, 269, 270, 291, 293, 296, 308, 314, 334, 336, 405,
    ],
    "POSE_LANDMARK_KEYS":       ["pose", "pose_landmarks"],
    "LEFT_HAND_LANDMARK_KEYS":  ["left_hand", "left_hand_landmarks"],
    "RIGHT_HAND_LANDMARK_KEYS": ["right_hand", "right_hand_landmarks"],
    "FACE_LANDMARK_KEYS":       ["face", "face_landmarks"],
}

assert CONFIG["NUM_KEYPOINTS"] * CONFIG["KEYPOINT_DIM"] == CONFIG["INPUT_DIM"]
assert len(CONFIG["SELECTED_FACE_INDICES"]) == 29
print("CONFIG OK")

In [ ]:
!pip install -q sentencepiece optuna --no-input

import os, json, zipfile, random, math, time, gc, logging, pathlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from transformers.modeling_outputs import BaseModelOutput
from tqdm.auto import tqdm
import optuna
from optuna.pruners import MedianPruner

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["OPTUNA_SEED"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
def _get_first_present(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    raise KeyError(f"None of {keys} found. Real keys: {list(d.keys())}")

def _landmarks_to_array(lm_list):
    if len(lm_list) == 0:
        return np.zeros((0, 2), dtype=np.float32)
    first = lm_list[0]
    if isinstance(first, dict):
        return np.array([[pt["x"], pt["y"]] for pt in lm_list], dtype=np.float32)
    return np.array([[pt[0], pt[1]] for pt in lm_list], dtype=np.float32)

def _extract_frames(raw_json):
    if isinstance(raw_json, list):
        return raw_json
    if isinstance(raw_json, dict):
        if "frames" in raw_json:
            return raw_json["frames"]
        for v in raw_json.values():
            if isinstance(v, list) and len(v) > 0:
                return v
    raise ValueError("Cannot find frame list in JSON.")

def select_keypoints(raw_frame):
    pose       = _landmarks_to_array(_get_first_present(raw_frame, CONFIG["POSE_LANDMARK_KEYS"]))
    left_hand  = _landmarks_to_array(_get_first_present(raw_frame, CONFIG["LEFT_HAND_LANDMARK_KEYS"]))
    right_hand = _landmarks_to_array(_get_first_present(raw_frame, CONFIG["RIGHT_HAND_LANDMARK_KEYS"]))
    face_full  = _landmarks_to_array(_get_first_present(raw_frame, CONFIG["FACE_LANDMARK_KEYS"]))
    if left_hand.shape[0]  == 0: left_hand  = np.zeros((21, 2), dtype=np.float32)
    if right_hand.shape[0] == 0: right_hand = np.zeros((21, 2), dtype=np.float32)
    face_sel = face_full[CONFIG["SELECTED_FACE_INDICES"]]
    combined = np.concatenate([pose, left_hand, right_hand, face_sel], axis=0)  # (104,2)
    return combined.reshape(-1).astype(np.float32)  # (208,)

def normalize_signspace(seq):
    T, flat = seq.shape
    K = flat // 2
    s = seq.reshape(T, K, 2)
    ls = s[:, CONFIG["LEFT_SHOULDER_IDX"],  :]
    rs = s[:, CONFIG["RIGHT_SHOULDER_IDX"], :]
    lh = s[:, CONFIG["LEFT_HIP_IDX"],       :]
    rh = s[:, CONFIG["RIGHT_HIP_IDX"],      :]
    smid = (ls + rs) / 2.0
    hmid = (lh + rh) / 2.0
    torso = np.linalg.norm(smid - hmid, axis=-1, keepdims=True)
    torso = np.clip(torso, 1e-6, None)
    centered = s - smid[:, None, :]
    return (centered / torso[:, None, :]).reshape(T, flat).astype(np.float32)

def pad_or_truncate(seq, max_len):
    T, flat = seq.shape
    if T >= max_len:
        return seq[:max_len].astype(np.float32), max_len
    padded = np.zeros((max_len, flat), dtype=np.float32)
    padded[:T] = seq
    return padded, T

print("Keypoint utilities loaded.")

In [ ]:
class YouTubeASLDataset(Dataset):
    def __init__(self, samples, zip_path, max_seq_len, tokenized_labels):
        """
        Lightweight dataset that takes a pre-built sample list.
        This way we build the index once and share it across all Optuna trials.
        """
        self.samples          = samples
        self.zip_path         = zip_path
        self.max_seq_len      = max_seq_len
        self._tokenized_labels = tokenized_labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        n = len(self.samples)
        for attempt in range(n):
            clip_id, _ = self.samples[(idx + attempt) % n]
            filename = f"{clip_id}.json"
            try:
                with zipfile.ZipFile(self.zip_path, "r") as zf:
                    with zf.open(filename) as fh:
                        raw_json = json.load(fh)
            except KeyError:
                continue
            except (zipfile.BadZipFile, json.JSONDecodeError):
                continue
            try:
                frames = _extract_frames(raw_json)
                if len(frames) == 0: continue
                per_frame  = [select_keypoints(f) for f in frames]
                keypoints  = np.stack(per_frame, axis=0)
                keypoints  = normalize_signspace(keypoints)
                keypoints, actual_len = pad_or_truncate(keypoints, self.max_seq_len)
            except Exception:
                continue
            mask = np.zeros(self.max_seq_len, dtype=np.float32)
            mask[:actual_len] = 1.0
            return {
                "input_ids":      torch.from_numpy(keypoints).float(),
                "attention_mask": torch.from_numpy(mask).float(),
                "labels":         torch.tensor(self._tokenized_labels[clip_id], dtype=torch.long),
                "clip_id":        clip_id,
            }
        raise RuntimeError("No valid clip found.")

print("Dataset class defined.")

In [ ]:
# Build the 10k clip index exactly once.
# Every Optuna trial uses this same fixed set — fair comparison.

print("Loading translations dictionary...")
with open(CONFIG["TRANSLATIONS_PATH"], "r", encoding="utf-8") as f:
    translations_raw = json.load(f)

all_samples = []
for video_id, video_data in translations_raw.items():
    for clip_id in video_data.get("clip_order", []):
        entry = video_data.get(clip_id)
        if entry and "translation" in entry:
            all_samples.append((clip_id, entry["translation"]))

print(f"Total clips with translations: {len(all_samples)}")

# Random sample of 10k — fixed seed for reproducibility
random.seed(CONFIG["OPTUNA_SEED"])
sampled = random.sample(all_samples, min(CONFIG["OPTUNA_SAMPLE_SIZE"], len(all_samples)))
print(f"Sampled for Optuna: {len(sampled)} clips")

# Pre-tokenize all translations ONCE (not per trial)
print("Pre-tokenizing translations (runs once)...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["MT5_MODEL_NAME"])
tokenized_labels = {}
for clip_id, text in tqdm(sampled, desc="Tokenizing"):
    enc = tokenizer(text, max_length=CONFIG["MAX_TARGET_LENGTH"],
                    truncation=True, padding=False)
    tokenized_labels[clip_id] = enc["input_ids"]

print(f"Tokenization complete. {len(tokenized_labels)} clips ready.")

# 90/10 split — fixed, shared across all trials
n_val   = max(1, int(0.1 * len(sampled)))
n_train = len(sampled) - n_val
train_samples = sampled[:n_train]
val_samples   = sampled[n_train:]
print(f"Train: {n_train} | Val: {n_val}")

In [ ]:
def collate_fn(batch):
    input_ids      = torch.stack([b["input_ids"]      for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    clip_ids       = [b["clip_id"] for b in batch]
    batch_max_len  = max(1, int(attention_mask.sum(dim=1).max().item()))
    input_ids      = input_ids[:, :batch_max_len, :]
    attention_mask = attention_mask[:, :batch_max_len]
    max_lbl        = max(len(b["labels"]) for b in batch)
    labels_pad     = torch.full((len(batch), max_lbl), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        labels_pad[i, :len(b["labels"])] = b["labels"]
    return {"input_ids": input_ids, "attention_mask": attention_mask,
            "labels": labels_pad, "clip_id": clip_ids}

print("collate_fn defined.")

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class SpatialTemporalEncoder(nn.Module):
    def __init__(self, input_dim, d_model, num_heads, num_layers, ffn_dim, dropout, max_len):
        super().__init__()
        self.proj    = nn.Linear(input_dim, d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=ffn_dim,
            dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
    def forward(self, x, mask):
        x = self.proj(x)
        x = self.pos_enc(x)
        x = self.transformer(x, src_key_padding_mask=(mask == 0))
        return x, mask

class SignLanguageTranslator(nn.Module):
    def __init__(self, input_dim, d_model, num_heads, num_layers, ffn_dim,
                 dropout, max_len, mt5_name):
        super().__init__()
        self.encoder  = SpatialTemporalEncoder(
            input_dim, d_model, num_heads, num_layers, ffn_dim, dropout, max_len)
        self.mt5      = T5ForConditionalGeneration.from_pretrained(mt5_name)
        mt5_hidden    = self.mt5.config.d_model
        self.proj_out = nn.Linear(d_model, mt5_hidden)
        # Freeze mT5 completely (Phase 3 rule)
        for p in self.mt5.parameters():
            p.requires_grad = False
    def forward(self, input_ids, attention_mask, labels=None):
        enc_hidden, enc_mask = self.encoder(input_ids, attention_mask)
        enc_hidden = self.proj_out(enc_hidden)
        return self.mt5(encoder_outputs=(enc_hidden,),
                        attention_mask=enc_mask, labels=labels)

print("Model classes defined.")

In [ ]:
def objective(trial):
    # ── 1. Sample hyperparameters ──────────────────────────────────────────
    lr           = trial.suggest_float("learning_rate",      5e-5,  5e-4, log=True)
    dropout      = trial.suggest_float("dropout",            0.05,  0.30)
    num_layers   = trial.suggest_categorical("num_encoder_layers", [2, 4, 6])
    ffn_dim      = trial.suggest_categorical("dim_feedforward",    [512, 1024, 2048])
    batch_size   = trial.suggest_categorical("batch_size",         [4, 8])
    grad_accum   = trial.suggest_categorical("grad_accum_steps",   [2, 4])
    # Warmup as a fraction of total steps — avoids warmup > total steps crash
    warmup_frac  = trial.suggest_float("warmup_fraction",    0.02,  0.10)

    # ── 2. Build DataLoaders ───────────────────────────────────────────────
    train_ds = YouTubeASLDataset(train_samples, CONFIG["ZIP_PATH"],
                                 CONFIG["MAX_SEQ_LEN"], tokenized_labels)
    val_ds   = YouTubeASLDataset(val_samples,   CONFIG["ZIP_PATH"],
                                 CONFIG["MAX_SEQ_LEN"], tokenized_labels)
    t_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=0, pin_memory=True,
                          collate_fn=collate_fn, drop_last=True)
    v_loader = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                          num_workers=0, pin_memory=True, collate_fn=collate_fn)
    if len(t_loader) == 0:
        raise optuna.TrialPruned()

    # ── 3. Scheduler math (proportional warmup) ────────────────────────────
    steps_per_epoch   = len(t_loader) // grad_accum
    total_steps       = max(1, steps_per_epoch * CONFIG["OPTUNA_EPOCHS"])
    warmup_steps      = max(1, int(warmup_frac * total_steps))
    trial.set_user_attr("warmup_steps_abs", warmup_steps)
    trial.set_user_attr("total_steps",      total_steps)

    # ── 4. Build model ─────────────────────────────────────────────────────
    set_seed(CONFIG["OPTUNA_SEED"])
    model = SignLanguageTranslator(
        input_dim  = CONFIG["INPUT_DIM"],
        d_model    = CONFIG["D_MODEL"],
        num_heads  = CONFIG["NUM_HEADS"],
        num_layers = num_layers,
        ffn_dim    = ffn_dim,
        dropout    = dropout,
        max_len    = CONFIG["MAX_SEQ_LEN"],
        mt5_name   = CONFIG["MT5_MODEL_NAME"],
    ).to(device)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=CONFIG["WEIGHT_DECAY"])
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps)
    scaler = GradScaler(enabled=CONFIG["FP16"])

    # ── 5. Train ───────────────────────────────────────────────────────────
    try:
        for epoch in range(CONFIG["OPTUNA_EPOCHS"]):
            model.train()
            optimizer.zero_grad()
            g_step = 0
            for step, batch in enumerate(t_loader):
                ids   = batch["input_ids"].to(device)
                amask = batch["attention_mask"].to(device)
                lbls  = batch["labels"].to(device)
                try:
                    with autocast(enabled=CONFIG["FP16"]):
                        out  = model(ids, amask, lbls)
                        loss = out.loss / grad_accum
                    scaler.scale(loss).backward()
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        torch.cuda.empty_cache()
                        optimizer.zero_grad()
                        continue
                    raise
                if (step + 1) % grad_accum == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        filter(lambda p: p.requires_grad, model.parameters()), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()
                    g_step += 1

            # ── Validate ───────────────────────────────────────────────────
            model.eval()
            val_losses = []
            with torch.no_grad():
                for batch in v_loader:
                    ids   = batch["input_ids"].to(device)
                    amask = batch["attention_mask"].to(device)
                    lbls  = batch["labels"].to(device)
                    with autocast(enabled=CONFIG["FP16"]):
                        out = model(ids, amask, lbls)
                    val_losses.append(out.loss.item())
            val_loss = float(np.mean(val_losses)) if val_losses else float("inf")
            print(f"  Trial {trial.number:>2} | Epoch {epoch+1}/{CONFIG["OPTUNA_EPOCHS"]} "
                  f"| val_loss={val_loss:.4f}")
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
    finally:
        # Free GPU before next trial — critical
        del model, optimizer, scheduler, scaler
        torch.cuda.empty_cache()
        gc.collect()

    return val_loss

print("Objective function defined. Ready to run study.")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# MedianPruner: kills a trial if its val_loss at any epoch
# is worse than the median of all completed trials at that epoch
pruner = MedianPruner(n_startup_trials=4, n_warmup_steps=1)
study  = optuna.create_study(direction="minimize", pruner=pruner)

print(f"Starting Optuna study: {CONFIG["OPTUNA_N_TRIALS"]} trials, "
      f"{CONFIG["OPTUNA_EPOCHS"]} epochs each, "
      f"{CONFIG["OPTUNA_SAMPLE_SIZE"]:,} clips\n")

study.optimize(objective, n_trials=CONFIG["OPTUNA_N_TRIALS"], show_progress_bar=True)

In [ ]:
best = study.best_trial

# Scale warmup back to Khizer's real 39k-clip / 15-epoch training run
# Real total steps ≈ (39000 * 0.9 / effective_batch) * 15
real_eff_batch  = best.params["batch_size"] * best.params["grad_accum_steps"]
real_steps      = int((39000 * 0.9 / real_eff_batch) * 15)
real_warmup     = max(1, int(best.params["warmup_fraction"] * real_steps))

print("=" * 65)
print("OPTUNA COMPLETE")
print("=" * 65)
print(f"Best val loss: {best.value:.4f}")
print(f"Trials completed: {len(study.trials)}")
print(f"Trials pruned:    {sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)}")
print()
print("─" * 65)
print("PASTE THESE VALUES INTO KHIZER's CONFIG (Cell 1):")
print("─" * 65)
print(f'  "LEARNING_RATE":            {best.params["learning_rate"]:.2e},')
print(f'  "DROPOUT":                  {best.params["dropout"]:.3f},')
print(f'  "NUM_ENCODER_LAYERS":       {best.params["num_encoder_layers"]},')
print(f'  "DIM_FEEDFORWARD":          {best.params["dim_feedforward"]},')
print(f'  "BATCH_SIZE":               {best.params["batch_size"]},')
print(f'  "GRADIENT_ACCUMULATION_STEPS": {best.params["grad_accum_steps"]},')
print(f'  "WARMUP_STEPS":             {real_warmup},  # scaled from warmup_fraction={best.params["warmup_fraction"]:.3f}')
print(f'  "MAX_EPOCHS":               15,  # unchanged — the 15-epoch trick')
print("─" * 65)